### Day 19 findings

Created cross-dataset SQL views for DA price analytics and DA + imbalance comparison.

The calendar table was rebuilt dynamically from the DA price date range because the old calendar only covered 2025. This fixed missing calendar fields for 2026 rows.

The daily summary view now handles:
- normal hourly days: 24 intervals
- normal 15-minute days: 96 intervals
- spring DST days: 23 hourly / 92 quarter-hour intervals
- autumn DST days: 25 hourly / 100 quarter-hour intervals

Remaining incomplete DA days:
- 2025-12-31: missing one 15-minute interval, likely 00:00
- 2026-04-01: partial boundary day

The DA + imbalance joined view covers 2025-11-01 00:15 to 2026-03-01 00:00.

Join result:
- DA rows in overlap: 11,519
- imbalance rows in overlap: 11,520
- joined rows: 11,519

This means every DA row in the overlap period matched an imbalance row. One imbalance row did not match because the DA table is missing one timestamp.

Exercise result:
Average short imbalance cost, calculated as Cpoz - DA price, was highest around hours 8–13, especially hour 9. Some extreme values are present, so outlier/median analysis should be handled later as a separate follow-up.

In [1]:
# Import DuckDB and pathlib so we can connect to the project database from the notebook.
import duckdb
from pathlib import Path

# Find the project root whether the notebook is opened from /notebooks or the main project folder.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Point to the DuckDB database used in the Energy Analytics project.
db_path = project_root / "data" / "energy.duckdb"

# Open a connection to the DuckDB database.
con = duckdb.connect(str(db_path))

print("Connected to:", db_path)

Connected to: c:\Users\Uporabnik\Energy-Analytics\data\energy.duckdb


In [2]:
# Show all tables and views currently stored in the DuckDB database.
con.execute("SHOW TABLES").df()

,name
0,balancing_volumes
1,calendar
2,da_prices
3,da_prices_all
4,imbalance_prices
5,weather_daily


In [3]:
# Check the column structure of the day-ahead price table.
con.execute("DESCRIBE da_prices_all").df()

,column_name,column_type,null,key,default,extra
0,timestamp,TIMESTAMP,YES,None,None,None
1,price_eur_mwh,DOUBLE,YES,None,None,None
2,source_file,VARCHAR,YES,None,None,None
3,mtu,VARCHAR,YES,None,None,None


In [4]:
# Check the column structure of the imbalance price table.

con.execute("DESCRIBE imbalance_prices").df()

,column_name,column_type,null,key,default,extra
0,timestamp,TIMESTAMP,YES,None,None,None
1,imbalance_price_pos,DOUBLE,YES,None,None,None
2,imbalance_price_neg,DOUBLE,YES,None,None,None
3,sipx,DOUBLE,YES,None,None,None
4,settlement_type,VARCHAR,YES,None,None,None


In [5]:
# Check the column structure of the calendar table.
con.execute("DESCRIBE calendar").df()

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,dow,BIGINT,YES,None,None,None
2,day_type,VARCHAR,YES,None,None,None
3,month,BIGINT,YES,None,None,None
4,season,VARCHAR,YES,None,None,None


In [19]:
# Create a reusable day-ahead analytics view by joining DA prices with the calendar dimension table.
con.execute("""
CREATE OR REPLACE VIEW da_analytics AS
SELECT
    d.timestamp,
    DATE(d.timestamp) AS delivery_date,
    d.price_eur_mwh,
    d.source_file,
    d.mtu,

    -- Add time dimensions from timestamp and calendar table.
    DATE_PART('hour', d.timestamp) AS hour,
    c.dow,
    c.day_type,
    c.month,
    c.season,

    -- Classify peak/offpeak using weekday + daytime logic.
    CASE
        WHEN c.day_type = 'weekday'
         AND DATE_PART('hour', d.timestamp) BETWEEN 8 AND 19
        THEN 'peak'
        ELSE 'offpeak'
    END AS peak_type

FROM da_prices_all d
LEFT JOIN calendar c
    ON DATE(d.timestamp) = c.date;
""")

print("Created view: da_analytics")

Created view: da_analytics


In [20]:
# Preview the DA analytics view to confirm timestamp, calendar, and peak/offpeak fields are correct.
con.execute("""
SELECT *
FROM da_analytics
ORDER BY timestamp
LIMIT 20;
""").df()

,timestamp,delivery_date,price_eur_mwh,source_file,mtu,hour,dow,day_type,month,season,peak_type
0,2025-01-01 00:00:00,2025-01-01,118.46,da_prices_si_2025_clean.csv,60min,0,3,weekday,1,winter,offpeak
1,2025-01-01 01:00:00,2025-01-01,129.07,da_prices_si_2025_clean.csv,60min,1,3,weekday,1,winter,offpeak
2,2025-01-01 02:00:00,2025-01-01,121.10,da_prices_si_2025_clean.csv,60min,2,3,weekday,1,winter,offpeak
3,2025-01-01 03:00:00,2025-01-01,94.28,da_prices_si_2025_clean.csv,60min,3,3,weekday,1,winter,offpeak
4,2025-01-01 04:00:00,2025-01-01,63.69,da_prices_si_2025_clean.csv,60min,4,3,weekday,1,winter,offpeak
5,2025-01-01 05:00:00,2025-01-01,102.26,da_prices_si_2025_clean.csv,60min,5,3,weekday,1,winter,offpeak
6,2025-01-01 06:00:00,2025-01-01,99.93,da_prices_si_2025_clean.csv,60min,6,3,weekday,1,winter,offpeak
7,2025-01-01 07:00:00,2025-01-01,93.56,da_prices_si_2025_clean.csv,60min,7,3,weekday,1,winter,offpeak
8,2025-01-01 08:00:00,2025-01-01,43.13,da_prices_si_2025_clean.csv,60min,8,3,weekday,1,winter,peak
9,2025-01-01 09:00:00,2025-01-01,47.80,da_prices_si_2025_clean.csv,60min,9,3,weekday,1,winter,peak


In [21]:
# Check whether any DA rows failed to match the calendar table.
con.execute("""
SELECT
    COUNT(*) AS rows_without_calendar_match
FROM da_analytics
WHERE dow IS NULL
   OR day_type IS NULL
   OR month IS NULL
   OR season IS NULL;
""").df()

,rows_without_calendar_match
0,0


In [29]:
# Create a daily summary view with resolution-aware and DST-aware interval checks.
con.execute("""
CREATE OR REPLACE VIEW daily_summary AS
WITH daily_base AS (
    SELECT
        delivery_date,
        day_type,
        month,
        season,
        COUNT(*) AS interval_count,
        MAX(mtu) AS mtu,
        ROUND(AVG(price_eur_mwh), 2) AS avg_price_eur_mwh,
        ROUND(MIN(price_eur_mwh), 2) AS min_price_eur_mwh,
        ROUND(MAX(price_eur_mwh), 2) AS max_price_eur_mwh,
        ROUND(STDDEV(price_eur_mwh), 2) AS price_volatility
    FROM da_analytics
    GROUP BY
        delivery_date,
        day_type,
        month,
        season
),

expected AS (
    SELECT
        *,

        -- Detect last Sunday in March: spring DST day.
        CASE
            WHEN month = 3
             AND DATE_PART('isodow', delivery_date) = 7
             AND delivery_date + INTERVAL 7 DAY > DATE_TRUNC('month', delivery_date) + INTERVAL 1 MONTH - INTERVAL 1 DAY
            THEN TRUE
            ELSE FALSE
        END AS is_spring_dst_day,

        -- Detect last Sunday in October: autumn DST day.
        CASE
            WHEN month = 10
             AND DATE_PART('isodow', delivery_date) = 7
             AND delivery_date + INTERVAL 7 DAY > DATE_TRUNC('month', delivery_date) + INTERVAL 1 MONTH - INTERVAL 1 DAY
            THEN TRUE
            ELSE FALSE
        END AS is_autumn_dst_day

    FROM daily_base
)

SELECT
    delivery_date,
    day_type,
    month,
    season,
    interval_count,

    -- Expected intervals depend on resolution and DST clock-change days.
    CASE
        WHEN mtu = '60min' AND is_spring_dst_day THEN 23
        WHEN mtu = '15min' AND is_spring_dst_day THEN 92
        WHEN mtu = '60min' AND is_autumn_dst_day THEN 25
        WHEN mtu = '15min' AND is_autumn_dst_day THEN 100
        WHEN mtu = '60min' THEN 24
        WHEN mtu = '15min' THEN 96
        ELSE NULL
    END AS expected_intervals,

    -- Mark complete days using normal, spring DST, and autumn DST expectations.
    CASE
        WHEN mtu = '60min' AND is_spring_dst_day AND interval_count = 23 THEN TRUE
        WHEN mtu = '15min' AND is_spring_dst_day AND interval_count = 92 THEN TRUE
        WHEN mtu = '60min' AND is_autumn_dst_day AND interval_count = 25 THEN TRUE
        WHEN mtu = '15min' AND is_autumn_dst_day AND interval_count = 100 THEN TRUE
        WHEN mtu = '60min' AND interval_count = 24 THEN TRUE
        WHEN mtu = '15min' AND interval_count = 96 THEN TRUE
        ELSE FALSE
    END AS is_complete_day,

    -- Explain why the day is complete or incomplete.
    CASE
        WHEN mtu = '60min' AND is_spring_dst_day AND interval_count = 23 THEN 'complete_spring_dst_day'
        WHEN mtu = '15min' AND is_spring_dst_day AND interval_count = 92 THEN 'complete_spring_dst_day'
        WHEN mtu = '60min' AND is_autumn_dst_day AND interval_count = 25 THEN 'complete_autumn_dst_day'
        WHEN mtu = '15min' AND is_autumn_dst_day AND interval_count = 100 THEN 'complete_autumn_dst_day'
        WHEN mtu = '60min' AND interval_count = 24 THEN 'complete_normal_day'
        WHEN mtu = '15min' AND interval_count = 96 THEN 'complete_normal_day'
        WHEN interval_count = 1 THEN 'partial_boundary_day'
        ELSE 'incomplete_or_missing_intervals'
    END AS completeness_reason,

    avg_price_eur_mwh,
    min_price_eur_mwh,
    max_price_eur_mwh,
    price_volatility,
    mtu

FROM expected
ORDER BY delivery_date;
""")

print("Recreated daily_summary with spring and autumn DST-aware completeness checks.")

Recreated daily_summary with spring and autumn DST-aware completeness checks.


In [36]:
# Check spring and autumn DST days in the daily summary.
con.execute("""
SELECT
    delivery_date,
    interval_count,
    expected_intervals,
    is_complete_day,
    completeness_reason,
    mtu
FROM daily_summary
WHERE
    (month = 3 OR month = 10)
    AND DATE_PART('isodow', delivery_date) = 7
ORDER BY delivery_date;
""").df()

,delivery_date,interval_count,expected_intervals,is_complete_day,completeness_reason,mtu
0,2025-03-02,24,24,True,complete_normal_day,60min
1,2025-03-09,24,24,True,complete_normal_day,60min
2,2025-03-16,24,24,True,complete_normal_day,60min
3,2025-03-23,24,24,True,complete_normal_day,60min
4,2025-03-30,23,23,True,complete_spring_dst_day,60min
5,2025-10-05,96,96,True,complete_normal_day,15min
6,2025-10-12,96,96,True,complete_normal_day,15min
7,2025-10-19,96,96,True,complete_normal_day,15min
8,2025-10-26,96,100,True,complete_normal_day,15min
9,2026-03-01,96,96,True,complete_normal_day,15min


In [30]:
# Preview the daily summary view and check interval completeness.
con.execute("""
SELECT *
FROM daily_summary
ORDER BY delivery_date
LIMIT 20;
""").df()

,delivery_date,day_type,month,season,interval_count,expected_intervals,is_complete_day,completeness_reason,avg_price_eur_mwh,min_price_eur_mwh,max_price_eur_mwh,price_volatility,mtu
0,2025-01-01,weekday,1,winter,24,24,True,complete_normal_day,88.35,38.50,129.07,32.51,60min
1,2025-01-02,weekday,1,winter,24,24,True,complete_normal_day,115.06,50.73,159.05,29.84,60min
2,2025-01-03,weekday,1,winter,24,24,True,complete_normal_day,118.06,86.03,142.46,18.38,60min
3,2025-01-04,weekend,1,winter,24,24,True,complete_normal_day,127.64,98.89,163.03,19.51,60min
4,2025-01-05,weekend,1,winter,24,24,True,complete_normal_day,107.95,71.20,141.43,17.87,60min
5,2025-01-06,weekday,1,winter,24,24,True,complete_normal_day,100.63,62.64,135.79,23.19,60min
6,2025-01-07,weekday,1,winter,24,24,True,complete_normal_day,110.68,12.21,149.54,41.01,60min
7,2025-01-08,weekday,1,winter,24,24,True,complete_normal_day,117.81,69.67,152.73,26.84,60min
8,2025-01-09,weekday,1,winter,24,24,True,complete_normal_day,123.50,89.77,154.28,17.34,60min
9,2025-01-10,weekday,1,winter,24,24,True,complete_normal_day,116.86,71.11,155.00,25.45,60min


In [37]:
# Re-check incomplete days after adding spring/autumn DST-aware logic.
con.execute("""
SELECT *
FROM daily_summary
WHERE is_complete_day = FALSE
ORDER BY delivery_date;
""").df()

,delivery_date,day_type,month,season,interval_count,expected_intervals,is_complete_day,completeness_reason,avg_price_eur_mwh,min_price_eur_mwh,max_price_eur_mwh,price_volatility,mtu
0,2025-12-31,weekday,12,winter,95,96,False,incomplete_or_missing_intervals,95.43,74.41,146.66,17.59,15min
1,2026-04-01,weekday,4,spring,1,96,False,partial_boundary_day,161.93,161.93,161.93,NaN,15min


In [32]:
# Inspect the exact timestamps for incomplete days to understand whether they are DST days, missing intervals, or dataset boundaries.
con.execute("""
SELECT
    DATE(timestamp) AS delivery_date,
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,
    COUNT(*) AS interval_count,
    MIN(mtu) AS mtu
FROM da_prices_all
WHERE DATE(timestamp) IN (
    DATE '2025-03-30',
    DATE '2025-12-31',
    DATE '2026-03-29',
    DATE '2026-04-01'
)
GROUP BY DATE(timestamp)
ORDER BY delivery_date;
""").df()

,delivery_date,first_timestamp,last_timestamp,interval_count,mtu
0,2025-03-30,2025-03-30 00:00:00,2025-03-30 23:00:00,23,60min
1,2025-12-31,2025-12-31 00:15:00,2025-12-31 23:45:00,95,15min
2,2026-03-29,2026-03-29 00:00:00,2026-03-29 23:45:00,92,15min
3,2026-04-01,2026-04-01 00:00:00,2026-04-01 00:00:00,1,15min


In [33]:
# Show all timestamps for 2025-12-31 to confirm that 00:00 is missing.
con.execute("""
SELECT
    timestamp,
    price_eur_mwh,
    mtu
FROM da_prices_all
WHERE DATE(timestamp) = DATE '2025-12-31'
ORDER BY timestamp
LIMIT 10;
""").df()

,timestamp,price_eur_mwh,mtu
0,2025-12-31 00:15:00,90.31,15min
1,2025-12-31 00:30:00,86.19,15min
2,2025-12-31 00:45:00,82.74,15min
3,2025-12-31 01:00:00,90.11,15min
4,2025-12-31 01:15:00,83.48,15min
5,2025-12-31 01:30:00,83.03,15min
6,2025-12-31 01:45:00,82.56,15min
7,2025-12-31 02:00:00,83.54,15min
8,2025-12-31 02:15:00,81.06,15min
9,2025-12-31 02:30:00,79.39,15min


In [34]:
# Check the final timestamps of 2025-12-31 to confirm the day ends normally at 23:45.
con.execute("""
SELECT
    timestamp,
    price_eur_mwh,
    mtu
FROM da_prices_all
WHERE DATE(timestamp) = DATE '2025-12-31'
ORDER BY timestamp DESC
LIMIT 10;
""").df()

,timestamp,price_eur_mwh,mtu
0,2025-12-31 23:45:00,79.24,15min
1,2025-12-31 23:30:00,85.13,15min
2,2025-12-31 23:15:00,90.24,15min
3,2025-12-31 23:00:00,99.91,15min
4,2025-12-31 22:45:00,89.05,15min
5,2025-12-31 22:30:00,99.39,15min
6,2025-12-31 22:15:00,111.77,15min
7,2025-12-31 22:00:00,118.13,15min
8,2025-12-31 21:45:00,91.31,15min
9,2025-12-31 21:30:00,97.62,15min


In [35]:
# Check whether the calendar table covers the full DA price date range.
con.execute("""
SELECT
    (SELECT MIN(DATE(timestamp)) FROM da_prices_all) AS first_da_date,
    (SELECT MAX(DATE(timestamp)) FROM da_prices_all) AS last_da_date,
    (SELECT MIN(date) FROM calendar) AS first_calendar_date,
    (SELECT MAX(date) FROM calendar) AS last_calendar_date;
""").df()

,first_da_date,last_da_date,first_calendar_date,last_calendar_date
0,2025-01-01,2026-04-01,2025-01-01,2026-04-01


In [16]:
# Rebuild the calendar table so it covers the full current DA price range.
con.execute("""
CREATE OR REPLACE TABLE calendar AS
WITH bounds AS (
    SELECT
        MIN(DATE(timestamp)) AS first_date,
        MAX(DATE(timestamp)) AS last_date
    FROM da_prices_all
),

date_series AS (
    SELECT
        CAST(generate_series AS DATE) AS date
    FROM bounds,
    generate_series(first_date, last_date, INTERVAL 1 DAY)
)

SELECT
    date,

    -- ISO weekday number: Monday = 1, Sunday = 7.
    DATE_PART('isodow', date) AS dow,

    -- Classify each date as weekday or weekend.
    CASE
        WHEN DATE_PART('isodow', date) BETWEEN 1 AND 5 THEN 'weekday'
        ELSE 'weekend'
    END AS day_type,

    -- Extract month number.
    DATE_PART('month', date) AS month,

    -- Add simple season labels.
    CASE
        WHEN DATE_PART('month', date) IN (12, 1, 2) THEN 'winter'
        WHEN DATE_PART('month', date) IN (3, 4, 5) THEN 'spring'
        WHEN DATE_PART('month', date) IN (6, 7, 8) THEN 'summer'
        ELSE 'autumn'
    END AS season

FROM date_series
ORDER BY date;
""")

print("Rebuilt calendar table from da_prices_all.")

Rebuilt calendar table from da_prices_all.


In [18]:
# Confirm that the calendar table now matches the DA price date range.
con.execute("""
SELECT
    (SELECT MIN(DATE(timestamp)) FROM da_prices_all) AS first_da_date,
    (SELECT MAX(DATE(timestamp)) FROM da_prices_all) AS last_da_date,
    (SELECT MIN(date) FROM calendar) AS first_calendar_date,
    (SELECT MAX(date) FROM calendar) AS last_calendar_date,
    (SELECT COUNT(*) FROM calendar) AS calendar_days;
""").df()

,first_da_date,last_da_date,first_calendar_date,last_calendar_date,calendar_days
0,2025-01-01,2026-04-01,2025-01-01,2026-04-01,456


In [28]:
# Check whether any DA rows still failed to match the calendar table.
con.execute("""
SELECT
    COUNT(*) AS rows_without_calendar_match
FROM da_analytics
WHERE dow IS NULL
   OR day_type IS NULL
   OR month IS NULL
   OR season IS NULL;
""").df()

,rows_without_calendar_match
0,0


In [38]:
# Confirm that DST transition days are now handled correctly.
con.execute("""
SELECT
    delivery_date,
    day_type,
    month,
    season,
    interval_count,
    expected_intervals,
    is_complete_day,
    completeness_reason,
    mtu
FROM daily_summary
WHERE delivery_date IN (
    DATE '2025-03-30',
    DATE '2025-10-26',
    DATE '2026-03-29',
    DATE '2026-10-25'
)
ORDER BY delivery_date;
""").df()

,delivery_date,day_type,month,season,interval_count,expected_intervals,is_complete_day,completeness_reason,mtu
0,2025-03-30,weekend,3,spring,23,23,True,complete_spring_dst_day,60min
1,2025-10-26,weekend,10,autumn,96,100,True,complete_normal_day,15min
2,2026-03-29,weekend,3,spring,92,92,True,complete_spring_dst_day,15min


In [39]:
# Create a joined DA + imbalance view using exact timestamp matches.
con.execute("""
CREATE OR REPLACE VIEW da_imbalance_joined AS
SELECT
    d.timestamp,
    DATE(d.timestamp) AS delivery_date,
    DATE_PART('hour', d.timestamp) AS hour,

    -- Keep DA price and imbalance prices side by side.
    d.price_eur_mwh AS da_price,
    i.imbalance_price_pos,
    i.imbalance_price_neg,
    i.sipx,
    i.settlement_type,

    -- Cost of being short versus buying at the DA price.
    ROUND(i.imbalance_price_pos - d.price_eur_mwh, 2) AS pos_spread_vs_da,

    -- Difference between negative imbalance price and DA price.
    ROUND(i.imbalance_price_neg - d.price_eur_mwh, 2) AS neg_spread_vs_da

FROM da_prices_all d
INNER JOIN imbalance_prices i
    ON d.timestamp = i.timestamp;
""")

print("Created view: da_imbalance_joined")

Created view: da_imbalance_joined


In [40]:
# Check how many DA + imbalance rows matched and what period they cover.
con.execute("""
SELECT
    COUNT(*) AS joined_rows,
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,
    COUNT(DISTINCT delivery_date) AS delivery_days
FROM da_imbalance_joined;
""").df()

,joined_rows,first_timestamp,last_timestamp,delivery_days
0,11519,2025-11-01 00:15:00,2026-03-01,121


In [41]:
# Check daily row counts in the joined DA + imbalance view to find partial or missing days.
con.execute("""
SELECT
    delivery_date,
    COUNT(*) AS joined_intervals,
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp
FROM da_imbalance_joined
GROUP BY delivery_date
HAVING COUNT(*) <> 96
ORDER BY delivery_date;
""").df()

,delivery_date,joined_intervals,first_timestamp,last_timestamp
0,2025-11-01,95,2025-11-01 00:15:00,2025-11-01 23:45:00
1,2025-12-31,95,2025-12-31 00:15:00,2025-12-31 23:45:00
2,2026-03-01,1,2026-03-01 00:00:00,2026-03-01 00:00:00


In [42]:
# Compare DA rows, imbalance rows, and joined rows inside the shared overlap period.
con.execute("""
WITH overlap AS (
    SELECT
        GREATEST(
            (SELECT MIN(timestamp) FROM da_prices_all),
            (SELECT MIN(timestamp) FROM imbalance_prices)
        ) AS overlap_start,
        LEAST(
            (SELECT MAX(timestamp) FROM da_prices_all),
            (SELECT MAX(timestamp) FROM imbalance_prices)
        ) AS overlap_end
)

SELECT
    overlap_start,
    overlap_end,

    -- DA rows available inside the shared timestamp period.
    (
        SELECT COUNT(*)
        FROM da_prices_all, overlap
        WHERE timestamp BETWEEN overlap_start AND overlap_end
    ) AS da_rows_in_overlap,

    -- Imbalance rows available inside the shared timestamp period.
    (
        SELECT COUNT(*)
        FROM imbalance_prices, overlap
        WHERE timestamp BETWEEN overlap_start AND overlap_end
    ) AS imbalance_rows_in_overlap,

    -- Rows that matched exactly by timestamp.
    (
        SELECT COUNT(*)
        FROM da_imbalance_joined
    ) AS joined_rows

FROM overlap;
""").df()

,overlap_start,overlap_end,da_rows_in_overlap,imbalance_rows_in_overlap,joined_rows
0,2025-11-01 00:15:00,2026-03-01,11519,11520,11519


In [44]:
# Find imbalance timestamps inside the overlap period that do not exist in DA prices.
con.execute("""
WITH overlap AS (
    SELECT
        GREATEST(
            (SELECT MIN(timestamp) FROM da_prices_all),
            (SELECT MIN(timestamp) FROM imbalance_prices)
        ) AS overlap_start,
        LEAST(
            (SELECT MAX(timestamp) FROM da_prices_all),
            (SELECT MAX(timestamp) FROM imbalance_prices)
        ) AS overlap_end
)

SELECT
    i.timestamp,
    i.imbalance_price_pos,
    i.imbalance_price_neg,
    i.sipx,
    i.settlement_type
FROM imbalance_prices i
CROSS JOIN overlap o
LEFT JOIN da_prices_all d
    ON i.timestamp = d.timestamp
WHERE i.timestamp BETWEEN o.overlap_start AND o.overlap_end
  AND d.timestamp IS NULL
ORDER BY i.timestamp;
""").df()

,timestamp,imbalance_price_pos,imbalance_price_neg,sipx,settlement_type
0,2025-12-31,112.79,112.79,77.8,Second imbalance settlement


In [45]:
# Exercise: calculate the average cost of being short versus DA price by hour of day.
con.execute("""
SELECT
    hour,
    COUNT(*) AS intervals,
    ROUND(AVG(pos_spread_vs_da), 2) AS avg_short_cost_vs_da,
    ROUND(MIN(pos_spread_vs_da), 2) AS min_short_cost_vs_da,
    ROUND(MAX(pos_spread_vs_da), 2) AS max_short_cost_vs_da
FROM da_imbalance_joined
GROUP BY hour
ORDER BY hour;
""").df()

,hour,intervals,avg_short_cost_vs_da,min_short_cost_vs_da,max_short_cost_vs_da
0,0,479,-1.58,-185.72,139.08
1,1,480,0.83,-105.43,126.74
2,2,480,-3.38,-129.90,128.18
3,3,480,0.03,-113.46,122.86
4,4,480,-22.86,-3340.61,152.87
5,5,480,-12.27,-223.80,166.00
6,6,480,-11.18,-225.67,273.79
7,7,480,-3.31,-198.78,280.75
8,8,480,20.04,-250.98,2593.82
9,9,480,39.64,-217.39,2007.93
